# Topic Modelling with Top Engagement Dataset


In [1]:
import torch, time, os, gc, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")
SESSION_START = time.time()

plt.style.use('seaborn-v0_8-colorblind')
CLEAN = Path('clean')

CUDA: True
GPU: Tesla T4
Memory: 15.6 GB
Start: 2026-08-17 13:33:45


In [2]:
bsky = pd.read_csv(CLEAN / 'bsky_top_engagement.csv')
truth = pd.read_csv(CLEAN / 'truth_posts_clean.csv', dtype={'post_id': str})
bsky['created_at'] = pd.to_datetime(bsky['created_at'], format='mixed', utc=True)
truth['created_at'] = pd.to_datetime(truth['created_at'], format='mixed', utc=True)
print(f"Bluesky (top engagement): {len(bsky):,} posts")
print(f"Truth Social (full):      {len(truth):,} posts")

Bluesky (top engagement): 76,557 posts
Truth Social (full):      48,086 posts


## Text Preprocessing for Topics

In [3]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))
URL_RE = re.compile(r'https?://\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#(\w+)')

def clean_for_topics(text):
    if not isinstance(text, str):
        return ''
    t = URL_RE.sub('', text)
    t = MENTION_RE.sub('', t)
    t = HASHTAG_RE.sub(r'\1', t)
    t = re.sub(r'[^a-zA-Z\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip().lower()
    tokens = [w for w in t.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

bsky['clean_text'] = bsky['text'].apply(clean_for_topics)
truth['clean_text'] = truth['text'].apply(clean_for_topics)

bsky = bsky[bsky['clean_text'].str.len() > 10].reset_index(drop=True)
truth = truth[truth['clean_text'].str.len() > 10].reset_index(drop=True)
print(f"After cleaning — Bluesky: {len(bsky):,}, Truth: {len(truth):,}")

After cleaning — Bluesky: 75,243, Truth: 47,127


## BERTopic Bluesky

In [4]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')


t0 = time.time()
bsky_embeddings = embed_model.encode(bsky['clean_text'].tolist(), batch_size=256, show_progress_bar=True)
print(f"Embedded in {(time.time()-t0)/60:.1f} min")

/home/defeo.k/dissertation-hpc-venv/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
Batches: 100%|██████████| 294/294 [00:30<00:00,  9.52it/s]


Embedded in 0.5 min


In [5]:

t0 = time.time()

topic_model_bsky = BERTopic(nr_topics='auto', verbose=True)
topics_bsky, probs_bsky = topic_model_bsky.fit_transform(bsky['clean_text'].tolist(), bsky_embeddings)

print(f"Topics found: {len(set(topics_bsky)) - 1}")
print(f"Fit in {(time.time()-t0)/60:.1f} min")
topic_model_bsky.get_topic_info().head(15)

2026-08-17 13:34:42,387 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-17 13:34:42,618 - BERTopic - Dimensionality - Completed ✓
2026-08-17 13:34:42,626 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-17 13:34:54,161 - BERTopic - Cluster - Completed ✓
2026-08-17 13:34:54,162 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-08-17 13:34:55,214 - BERTopic - Representation - Completed ✓
2026-08-17 13:34:55,215 - BERTopic - Topic reduction - Reducing number of topics
2026-08-17 13:34:55,269 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-17 13:34:56,199 - BERTopic - Representation - Completed ✓
2026-08-17 13:34:56,207 - BERTopic - Topic reduction - Reduced number of topics from 136 to 73


Topics found: 72
Fit in 0.2 min


,Topic,Count,Name,Representation,Representative_Docs
0,-1,69820,-1_chatgpt_genai_like_people,"[chatgpt, genai, like, people, slop, midjourne...",[pls use looks like substitute every instance ...
1,0,2099,0_chatgpt_asked_ask_using,"[chatgpt, asked, ask, using, use, answer, like...","[chatgpt people, like chatgpt, chatgpt like]"
2,1,811,1_anti_people_slop_genai,"[anti, people, slop, genai, com, google, new, ...",[terrible idea want slop search results gonna ...
3,2,294,2_digitalart_midjourneyai_midjourneyart_aigene...,"[digitalart, midjourneyai, midjourneyart, aige...",[aiart midjourney digitalart generativeai gene...
4,3,189,3_aicommunity_aiartcommunity_aiart_midjourney,"[aicommunity, aiartcommunity, aiart, midjourne...","[aiartcommunity aiart aicommunity midjourney, ..."
5,4,184,4_midjourney_aiart_caturday_sweet,"[midjourney, aiart, caturday, sweet, dreams, s...","[midjourney aiart, midjourney aiart, midjourne..."
6,5,147,5_dilf_daddybear_oldman_silverdaddy,"[dilf, daddybear, oldman, silverdaddy, chubby,...",[nsfw bear grandpa silverdaddy fat chubby dadd...
7,6,106,6_creativeai_generativeart_synthart_aiart,"[creativeai, generativeart, synthart, aiart, g...",[synthart genai creativeai aiart generativeart...
8,7,92,7_blackpeople_percent_addtoblacksky_anime,"[blackpeople, percent, addtoblacksky, anime, b...",[goodnight family digitalart generativeai gene...
9,8,90,8_anime_wallpaper_generativeai_digitalart,"[anime, wallpaper, generativeai, digitalart, m...",[cyber projects anime manga aiart midjourney d...


In [6]:
# topics over time
tot_bsky = topic_model_bsky.topics_over_time(
    bsky['clean_text'].tolist(), bsky['created_at'].tolist(), nr_bins=50
)
topic_model_bsky.visualize_topics_over_time(tot_bsky, top_n_topics=10)

50it [00:10,  4.93it/s]


In [7]:
# save
topic_model_bsky.save(str(CLEAN / 'bsky_top_bertopic_model'))
bsky['topic'] = topics_bsky
bsky[['post_uri', 'topic']].to_csv(CLEAN / 'bsky_top_topics.csv', index=False)
print("Bluesky model saved")

2026-08-17 13:35:08,811 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Bluesky model saved


## BERTopic Truth Social

In [8]:

t0 = time.time()
truth_embeddings = embed_model.encode(truth['clean_text'].tolist(), batch_size=256, show_progress_bar=True)
print(f"Embedded in {(time.time()-t0)/60:.1f} min")

Batches: 100%|██████████| 185/185 [00:44<00:00,  4.16it/s]


Embedded in 0.8 min


In [9]:

t0 = time.time()

topic_model_truth = BERTopic(nr_topics='auto', verbose=True)
topics_truth, probs_truth = topic_model_truth.fit_transform(truth['clean_text'].tolist(), truth_embeddings)

print(f"Topics found: {len(set(topics_truth)) - 1}")
print(f"Fit in {(time.time()-t0)/60:.1f} min")
topic_model_truth.get_topic_info().head(15)

2026-08-17 13:35:55,029 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-17 13:35:55,118 - BERTopic - Dimensionality - Completed ✓
2026-08-17 13:35:55,121 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-17 13:35:59,198 - BERTopic - Cluster - Completed ✓
2026-08-17 13:35:59,199 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-08-17 13:36:01,127 - BERTopic - Representation - Completed ✓
2026-08-17 13:36:01,130 - BERTopic - Topic reduction - Reducing number of topics
2026-08-17 13:36:01,319 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-17 13:36:03,193 - BERTopic - Representation - Completed ✓
2026-08-17 13:36:03,200 - BERTopic - Topic reduction - Reduced number of topics from 351 to 188


Topics found: 187
Fit in 0.2 min


,Topic,Count,Name,Representation,Representative_Docs
0,-1,34117,-1_apos_bigtech_quot_artificialintelligence,"[apos, bigtech, quot, artificialintelligence, ...",[wake something much bigger covid happening ri...
1,0,2908,0_jan_weareback_panamacanal_hostages,"[jan, weareback, panamacanal, hostages, liars,...",[truth trump biden bidencrimefamily democrats ...
2,1,637,1_securities_artificialintelligence_bonds_fina...,"[securities, artificialintelligence, bonds, fi...",[analysts warn despite optimism year end rally...
3,2,636,2_mrnavaccines_reference_updated_conservative,"[mrnavaccines, reference, updated, conservativ...",[updated conservative article reference list t...
4,3,587,3_mandatorymasks_massformation_ivermectin_noam...,"[mandatorymasks, massformation, ivermectin, no...",[updated covid mrna vaccine article reference ...
5,4,446,4_bitcoin_fixthemoney_bitcoinnews_bitcoinprice,"[bitcoin, fixthemoney, bitcoinnews, bitcoinpri...",[psstpssst clown car bitcoin btc hucksters pre...
6,5,410,5_prisons_brazil_dominate_worldwide,"[prisons, brazil, dominate, worldwide, lula, e...",[hidden news comey turn state remember statute...
7,6,370,6_church_people_immigration_churches,"[church, people, immigration, churches, foster...",[folks realize speed amp efficiency doge techi...
8,7,311,7_illegals_accelerating_amp_rubio,"[illegals, accelerating, amp, rubio, deported,...",[goldenage futureism rubio ice vance bondi noe...
9,8,263,8_sanctity_concepts_revenge_enemies,"[sanctity, concepts, revenge, enemies, embrace...",[war hate revenge censorship enemies peace amp...


In [10]:
tot_truth = topic_model_truth.topics_over_time(
    truth['clean_text'].tolist(), truth['created_at'].tolist(), nr_bins=30
)
topic_model_truth.visualize_topics_over_time(tot_truth, top_n_topics=10)

30it [00:10,  2.73it/s]


In [11]:
topic_model_truth.save(str(CLEAN / 'truth_top_bertopic_model'))
truth['topic'] = topics_truth
truth[['post_id', 'topic']].to_csv(CLEAN / 'truth_top_topics.csv', index=False)
print("Truth Social model saved")

2026-08-17 13:36:15,597 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Truth Social model saved


## Topic + MBFC Overlay

In [15]:
bsky = bsky.drop(columns=[c for c in bsky.columns if 'topic' in c], errors='ignore')
bsky = bsky.merge(topics_df, on='post_uri', how='left')
print(f"Topics loaded: {bsky['topic'].notna().sum():,} posts with topics")

Topics loaded: 75,243 posts with topics


In [17]:
topics_df = pd.read_csv(CLEAN / 'bsky_top_topics.csv')
bsky = bsky.merge(topics_df, on='post_uri', how='left')
print(f"Topics loaded: {bsky['topic'].notna().sum():,} posts with topics")

Topics loaded: 75,243 posts with topics


In [18]:
from urllib.parse import urlparse

mbfc_path = '/home/defeo.k/kdefeo/Factual-Reporting-and-Political-Bias-Web-Interactions/data/mbfc.csv'
if os.path.exists(mbfc_path):
    mbfc = pd.read_csv(mbfc_path)
    mbfc_lookup = dict(zip(mbfc['source'], mbfc[['bias', 'factual_reporting']].to_dict('records')))

    def extract_domain(url):
        try:
            domain = urlparse(url).netloc.lower()
            return domain[4:] if domain.startswith('www.') else domain
        except:
            return None

    def get_factuality(text):
        urls = re.findall(r'https?://[^\s<>"{}|\\^`\[\]]+', str(text))
        for url in urls:
            d = extract_domain(url)
            if d and d in mbfc_lookup:
                return mbfc_lookup[d].get('factual_reporting')
        return None

    bsky['mbfc_factuality'] = bsky['text'].apply(get_factuality)
    topic_fact = bsky[bsky['mbfc_factuality'].notna()].groupby('topic')['mbfc_factuality'].value_counts().unstack(fill_value=0)
    topic_fact['total'] = topic_fact.sum(axis=1)
    print("Bluesky — Topics x Factuality:")
    print(topic_fact.sort_values('total', ascending=False).head(15).to_string())
else:
    print(f"MBFC file not found at {mbfc_path}")
    print("Clone the repo: git clone https://github.com/idiap/Factual-Reporting-and-Political-Bias-Web-Interactions.git")

Bluesky — Topics x Factuality:
mbfc_factuality  high  mixed  total
topic                              
-1                 77     22     99
 1                  3      0      3
 0                  1      0      1


## Cross-Platform Topic Comparison

In [19]:
print("BLUESKY TOP 10 TOPICS")
print(topic_model_bsky.get_topic_info().head(11).to_string())
print()
print("TRUTH SOCIAL TOP 10 TOPICS")
print(topic_model_truth.get_topic_info().head(11).to_string())

BLUESKY TOP 10 TOPICS
    Topic  Count                                                 Name                                                                                                                        Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               Representative_Docs
0      -1  69820                         -1_chatgpt_genai_like_people                                                             [chatgpt, genai, l

## Session Summary

In [20]:
elapsed = (time.time() - SESSION_START) / 3600

print("TOPIC MODELLING SESSION SUMMARY")
print(f"  Hardware: {torch.cuda.get_device_name(0)}")
print(f"  Bluesky: {len(bsky):,} posts, {len(set(topics_bsky))-1} topics")
print(f"  Truth Social: {len(truth):,} posts, {len(set(topics_truth))-1} topics")
print(f"  Total time: {elapsed:.2f} hours")
print(f"  End: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print()
for f in sorted(CLEAN.glob('*_top_*.csv')):
    print(f"  {f.name}: {f.stat().st_size/1e6:.1f} MB")

TOPIC MODELLING SESSION SUMMARY
  Hardware: Tesla T4
  Bluesky: 75,243 posts, 72 topics
  Truth Social: 47,127 posts, 187 topics
  Total time: 0.53 hours
  End: 2026-08-17 14:05:24

  bsky_top_emotions.csv: 6.6 MB
  bsky_top_engagement.csv: 45.6 MB
  bsky_top_sentiment.csv: 6.1 MB
  bsky_top_topics.csv: 5.6 MB
  truth_top_emotions.csv: 1.7 MB
  truth_top_sentiment.csv: 1.3 MB
  truth_top_topics.csv: 1.0 MB
